<a href="https://colab.research.google.com/github/Prateeksandhu61/LAPD-Crime-Case-Dataset-/blob/main/week2_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gspread==6.0.0

In [ ]:
!pip install pytrends gspread gspread_dataframe

In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
wb_id = "1rv9ae8Eu1EQDRMRvpbDKk_Uw_mth8qzolssHs_5iLz8"
sheet_id_source = 0

wb = gc.open_by_key(wb_id)
sheet = wb.get_worksheet(sheet_id_source)
data_dictionary = sheet.get_all_records()
print(data_dictionary)


[{'keywords': 'Comedy'}, {'keywords': 'Dance'}, {'keywords': 'Fashion'}, {'keywords': 'Festivals'}, {'keywords': 'Social Media'}, {'keywords': 'Indigenous Voices'}, {'keywords': 'Language'}, {'keywords': 'Literature'}, {'keywords': 'Concerts'}, {'keywords': 'Movies'}, {'keywords': 'Museums'}, {'keywords': 'Music'}, {'keywords': 'Photography'}, {'keywords': 'Pop Culture'}, {'keywords': 'Sports'}, {'keywords': 'Streaming'}, {'keywords': 'Television'}, {'keywords': 'Theater'}, {'keywords': 'Visual Arts'}, {'keywords': 'Arts and Culture'}, {'keywords': 'Lifestyle'}, {'keywords': 'Cars'}, {'keywords': 'Food'}, {'keywords': 'Health'}, {'keywords': 'Parenting'}, {'keywords': 'Pets'}, {'keywords': 'Real Estate'}, {'keywords': 'Relationships'}, {'keywords': 'Religion'}, {'keywords': 'Robotics'}, {'keywords': 'Travel'}, {'keywords': 'Vedio Games'}, {'keywords': 'Computers'}, {'keywords': 'Agriculture'}, {'keywords': 'Animals'}, {'keywords': 'Environment'}, {'keywords': 'Science'}, {'keywords': '

In [ ]:
GOOGLE_SHEETS_CREDS_FILE = "path/to/your/credentials.json"
SPREADSHEET_NAME = "Google Trends Data"

In [ ]:
from pytrends.request import TrendReq
from datetime import datetime
import pandas as pd
import time

# --- ADDED CODE START ---
# Define a custom User-Agent to make the request look like a real browser
requests_args = {
    'headers': {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }
}

pytrend = TrendReq(hl='en-US', tz=300)

sheet_iot_name = "iot_data"
sheet_related_name = "related_data"

try:
    iot_ws = wb.worksheet(sheet_iot_name)
except:
    iot_ws = wb.add_worksheet(title=sheet_iot_name, rows="1000", cols="20")

try:
    related_ws = wb.worksheet(sheet_related_name)
except:
    related_ws = wb.add_worksheet(title=sheet_related_name, rows="1000", cols="20")

iot_list = []
related_list = []

regions = {
    "Canada": "CA"
}

for idx, kw_record in enumerate(data_dictionary):
    kw = kw_record['keywords']
    print(f"[{idx+1}/{len(data_dictionary)}] Processing keyword: {kw}")

    for region_name, geo_code in regions.items():
        print(f"  Fetching for region: {region_name} ({geo_code})")

        try:
            # iot
            pytrends.build_payload([kw], timeframe="today 3-m", geo=geo_code)
            iot = pytrends.interest_over_time()

            if not iot.empty:
                iot.reset_index(inplace=True)
                iot["value"] = iot[kw]
                iot = iot[["date", "value", "isPartial"]]
                iot["keyword"] = kw
                iot["geo"] = geo_code
                iot["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                iot_list.append(iot.tail(5))

            time.sleep(2)

            # related queries
            related = pytrends.related_queries()
            if related.get(kw):
                rising = related[kw]["rising"]
                if rising is not None:
                    rising["keyword"] = kw
                    rising["geo"] = geo_code
                    rising["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    related_list.append(rising.head(5))

            print(f"  Success for '{kw}' in {region_name}")

        except Exception as e:
            print(f"  Error for '{kw}' in {region_name}: {e}")
            if "429" in str(e):
                print("  Rate limited! Waiting 60 seconds...")
                time.sleep(60)

        time.sleep(5)

# --- ✅ WRITE TO GOOGLE SHEETS (Append without deleting old data) ---

# Write IoT data to Google Sheet
if iot_list:
    iot_df = pd.concat(iot_list, ignore_index=True)
    if 'date' in iot_df.columns:
        iot_df['date'] = pd.to_datetime(iot_df['date']).dt.strftime('%Y-%m-%d')

    existing_rows = len(iot_ws.get_all_values())
    if existing_rows == 0:
        # If sheet is empty, add header
        iot_ws.append_row(iot_df.columns.tolist())

    # Append new rows below existing data
    iot_ws.append_rows(iot_df.astype(str).values.tolist())
    print("\nIoT data appended to existing sheet")
else:
    print("\nNo IoT data fetched")

# Write Related Queries data to Google Sheet
if related_list:
    related_df = pd.concat(related_list, ignore_index=True)

    existing_rows = len(related_ws.get_all_values())
    if existing_rows == 0:
        related_ws.append_row(related_df.columns.tolist())

    related_ws.append_rows(related_df.astype(str).values.tolist())
    print("Related Queries appended to existing sheet")
else:
    print("No Related Queries fetched")

print("\nAll data processing complete!")


[1/41] Processing keyword: Comedy
  Fetching for region: Canada (CA)
  Error for 'Comedy' in Canada: The request failed: Google returned a response with code 429
  Rate limited! Waiting 60 seconds...
[2/41] Processing keyword: Dance
  Fetching for region: Canada (CA)
  Error for 'Dance' in Canada: The request failed: Google returned a response with code 429
  Rate limited! Waiting 60 seconds...
[3/41] Processing keyword: Fashion
  Fetching for region: Canada (CA)
  Error for 'Fashion' in Canada: The request failed: Google returned a response with code 429
  Rate limited! Waiting 60 seconds...
[4/41] Processing keyword: Festivals
  Fetching for region: Canada (CA)
  Error for 'Festivals' in Canada: The request failed: Google returned a response with code 429
  Rate limited! Waiting 60 seconds...
[5/41] Processing keyword: Social Media
  Fetching for region: Canada (CA)
  Error for 'Social Media' in Canada: The request failed: Google returned a response with code 429
  Rate limited! Wait

In [ ]:
def write_to_sheet(ws, df, expected_columns):
    """
    Appends a DataFrame to a Google Sheet worksheet.
    Adds headers if the worksheet is empty.


    Args:
        ws: gspread worksheet object
        df: pandas DataFrame
        expected_columns: list of column names to enforce order
    """
    if df.empty:
        print(f" No data to append for {ws.title}")
        return

    # Ensure only expected columns, in correct order
    df = df[[c for c in expected_columns if c in df.columns]]

    # Convert everything to string (Google Sheets safe)
    df = df.astype(str)

    # Add headers if sheet is empty
    if len(ws.get_all_values()) == 0:
        ws.append_row(df.columns.tolist())

    # Append rows
    ws.append_rows(df.values.tolist())
    print(f" Data appended to {ws.title}")


In [ ]:
# Convert lists to DataFrames and append
if iot_list:
    iot_df = pd.concat(iot_list, ignore_index=True)

    if 'date' in iot_df.columns:
        iot_df['date'] = iot_df['date'].dt.strftime('%Y-%m-%d')

    # Convert to list
    iot_data = iot_df.astype(str).values.tolist()

    # Write header first
    iot_ws.clear()
    iot_ws.append_row(iot_df.columns.tolist())
    iot_ws.append_rows(iot_data)

    print("\n IoT data with header appended")
else:
    print("\n No IoT data fetched")


if related_list:
    related_df = pd.concat(related_list, ignore_index=True)

    # Convert to list
    related_data = related_df.astype(str).values.tolist()

    # Write header first
    related_ws.clear()
    related_ws.append_row(related_df.columns.tolist())
    related_ws.append_rows(related_data)

    print(" Related queries with header appended")
else:
    print(" No Related queries fetched")
